# mam-graph demo
Force-directed graphs rendered with d3 inside JupyterLab.

In [12]:
%load_ext mam_graph

The mam_graph extension is already loaded. To reload it, use:
  %reload_ext mam_graph


## Render a variable from an earlier cell
Define a graph in one cell, then show it by name with `%forcegraph`.

In [13]:
import networkx as nx

G = nx.karate_club_graph()
for _, d in G.nodes(data=True):
    d["group"] = d["club"]

In [14]:
%forcegraph G --labels --charge -200 --height 600

<ForceGraph: 34 nodes, 78 links>

Plain dicts work too (`links` or `edges`, as produced by `nx.node_link_data`):

In [15]:
ring = {
    "nodes": [{"id": i, "group": i % 4} for i in range(30)],
    "links": [{"source": i, "target": (i * 7 + 3) % 30} for i in range(30)],
}

In [16]:
%forcegraph ring --labels

<ForceGraph: 30 nodes, 30 links>

In [17]:
D = nx.gn_graph(25, seed=1)

In [18]:
%forcegraph D --link-distance 60  # directed graphs get arrows

<ForceGraph: 25 nodes, 24 links>

## Python API
`ForceGraph(...)`, `ForceGraph.from_networkx(...)` and `ForceGraph.from_data(...)` return displayable objects.

In [19]:
from mam_graph import ForceGraph

ForceGraph.from_data(ring, show_labels=True)

<ForceGraph: 30 nodes, 30 links>

## Hierarchical edge bundling
Leaves sit on a circle, grouped by a hierarchy; links are bundled along it. `--levels` builds the hierarchy from node attributes (outermost first). Hover a node to highlight its links (red = outgoing, blue = incoming).

In [20]:
import random

random.seed(1)
areas = ["V1", "V2", "V4", "MT", "TEO", "LIP", "FEF", "46"]
layers = ["II/III", "IV", "V", "VI"]

M = nx.DiGraph()
for area in areas:
    for layer in layers:
        for pop in "EI":
            M.add_node(
                f"{area} {layer} {pop}",
                area=area,
                layer=layer,
                label=f"{area} {layer} {pop}",
            )
nodes = list(M)
for _ in range(250):
    M.add_edge(*random.sample(nodes, 2), weight=random.choice([1, 1, 2, 4]))

In [21]:
%edgebundle M --levels area layer --color-links

<EdgeBundle: 64 leaves, 241 links>

Tune bundling strength with `--beta` (0 = straight lines, 1 = fully bundled):

In [22]:
%edgebundle M --levels area --beta 0.5 --no-labels --size 400

<EdgeBundle: 64 leaves, 241 links>

A mapping with explicit `parent` ids works too, as does the Python API:

In [23]:
from mam_graph import EdgeBundle

tree = {
    "nodes": [
        {"id": "brain"},
        {"id": "visual", "parent": "brain"},
        {"id": "motor", "parent": "brain"},
        *({"id": a, "parent": "visual"} for a in ["V1", "V2", "V4", "MT"]),
        *({"id": a, "parent": "motor"} for a in ["M1", "PMd", "SMA"]),
    ],
    "links": [
        {"source": s, "target": t}
        for s, t in [
            ("V1", "V2"),
            ("V2", "V4"),
            ("V1", "MT"),
            ("MT", "PMd"),
            ("PMd", "M1"),
            ("SMA", "M1"),
            ("V4", "SMA"),
        ]
    ],
}
EdgeBundle.from_data(tree, directed=True, size=400)

<EdgeBundle: 7 leaves, 7 links>

Run `%forcegraph?` or `%edgebundle?` for all options.